In the previous notebook models_statenisland.ipynb, we found that the Prophet model performed well in terms of forecasting rat sightings by day citywide. In this notebook, we will do some more feature engineering and hyperparameter tuning to obtain a better optimal model.

Because we wish this to be reusable, we will write things for Prophet and NeuralProphet separately. 

# Import Packages

In [1]:
!pip install optuna

!pip install plotly ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
from prophet.plot import plot_plotly, plot_components_plotly
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import optuna
from neuralprophet import NeuralProphet
import logging


/Users/yaeleisenberg/micromamba/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


# Prophet

## Import the data

In [2]:
# set up the time series split
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Prepare Prophet

In [3]:
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [4]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [5]:
rs_saved = rs.copy()
df = rs.copy()

## Optuna for Hyperparameter Tuning for Prophet

In [6]:
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [7]:
# This code block is a grid search for hyperparameters.
# To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
# So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = '2054 days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1],
    'seasonality_prior_scale': [5],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

15:10:24 - cmdstanpy - INFO - Chain [1] start processing
15:10:24 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/13 [00:00<?, ?it/s]

15:10:24 - cmdstanpy - INFO - Chain [1] start processing
15:10:24 - cmdstanpy - INFO - Chain [1] done processing
15:10:24 - cmdstanpy - INFO - Chain [1] start processing
15:10:24 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1] done processing
15:10:25 - cmdstanpy - INFO - Chain [1] start processing
15:10:25 - cmdstanpy - INFO - Chain [1]

In [8]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,14 days,1.0289,1.0143,0.8053,0.5393,0.4349,0.4385,0.9694


## Train the Model

In [9]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)

15:10:26 - cmdstanpy - INFO - Chain [1] start processing
15:10:26 - cmdstanpy - INFO - Chain [1] done processing


## Plots and Evaluation of the Model

In [10]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()

# Neural Prophet

In [11]:
np.NaN = np.nan


# the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# the errors that do show up are not all that important and a lot is due to outdated packages.
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("neuralprophet").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("NP").setLevel(logging.ERROR)

In [12]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

rs.head()

,ds,y
4,2020-01-01,1
9,2020-01-02,1
14,2020-01-03,1
27,2020-01-06,4
32,2020-01-07,2


In [13]:
## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [14]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)


regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

In [15]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=14)


def objective(trial):
    regressor_lags = {
        'apparent_temperature_max': trial.suggest_int('lag_temp_max', 1, 60),
        'apparent_temperature_min': trial.suggest_int('lag_temp_min', 1, 60),
        'snowfall_sum': trial.suggest_int('lag_snowfall', 1, 7),
    }
    n_lags = trial.suggest_int('n_lags', 1, 60)
    epochs = trial.suggest_int('epochs', 10, 250)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1, log=True)
    batch_size = trial.suggest_int('batch_size', 12, 248)
    ar_reg = trial.suggest_float('ar_reg', 0.5, 3)
    fold_rmses = []
    for i, (train_idx, test_idx) in enumerate(tscv.split(rs)):

        train = rs.iloc[train_idx].copy()
        test = rs.iloc[test_idx].copy()
        
        existing_regressors = [col for col in regressed_features if col in train.columns]
        train = train.dropna(subset=["y"] + existing_regressors)
        test = test.dropna(subset=existing_regressors)
        
        # Skip fold if too few rows
        if len(train) < 20 or len(test) < 1:
            continue
        
        model = NeuralProphet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            n_lags=n_lags,
            epochs=epochs,
            ar_reg = ar_reg,
            accelerator="auto",   # uses GPU if available
            learning_rate=learning_rate,
            batch_size=batch_size
        )
        model.add_country_holidays(country_name="US")
        for col in existing_regressors:
            model.add_lagged_regressor(col, n_lags=regressor_lags[col])
        
        model.fit(train, freq="D", progress="off")
        future = pd.concat([
            train[['ds','y'] + existing_regressors],
            test[['ds','y']].merge(wd[['ds'] + existing_regressors], on="ds", how="left")
        ])
        future = future.dropna(subset=existing_regressors)
        forecast = model.predict(future)
        
        y_pred = forecast["yhat1"].iloc[-len(test):].values
        y_true = test["y"].values
        
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(rmse)

        intermediate_score = np.mean(fold_rmses)
        trial.report(intermediate_score, i)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
    return np.mean(fold_rmses) if fold_rmses else float("inf")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)  # adjust n_trials as needed



best_params = study.best_params

print("Best Parameters", best_params)
print("Best RMSE:", study.best_value)

[I 2026-03-10 15:10:28,873] A new study created in memory with name: no-name-5858e0b5-1a10-4612-a0da-9380c5a64368


Training: 0it [00:00, ?it/s]

Predicting: 9it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 10it [00:00, ?it/s]

...

Predicting: 15it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 15it [00:00, ?it/s]

[I 2026-03-10 15:17:51,897] Trial 14 finished with value: 0.7794634857913697 and parameters: {'lag_temp_max': 38, 'lag_temp_min': 21, 'lag_snowfall': 5, 'n_lags': 12, 'epochs': 40, 'learning_rate': 0.28003125204832974, 'batch_size': 152, 'ar_reg': 1.6177878387022375}. Best is trial 13 with value: 0.7533379003307871.


Best Parameters {'lag_temp_max': 35, 'lag_temp_min': 21, 'lag_snowfall': 4, 'n_lags': 14, 'epochs': 50, 'learning_rate': 0.2834689538082329, 'batch_size': 153, 'ar_reg': 1.6574629887750523}
Best RMSE: 0.7533379003307871


Parameters: {'lag_temp_max': 54, 'lag_temp_min': 18, 'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923}. Best is trial 83 with value: 2.9466650941279124.


In [16]:
# best_params = dict({'lag_temp_max': 54, 'lag_temp_min': 18, 
#                     'lag_snowfall': 1, 'n_lags': 59, 'epochs': 493, 'learning_rate': 0.003214767890388168, 'batch_size': 220, 'ar_reg': 0.5847571241076923})

In [17]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

## Restrict to STATEN ISLAND

rs = rs[rs['borough']=='STATEN ISLAND']

## Drop the column with borough

rs = rs.drop(columns=['borough'])

## rename columns for prophet

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)


## Add weather data.
import requests

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")

## Evaluate the Model

In [18]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min','snowfall_sum']


wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

rs = rs.merge(
    wd[['ds'] + regressed_features],
    on="ds",
    how="left"
)

lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):

    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()


    model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          n_lags= best_params['n_lags'],
                          ar_reg=best_params['ar_reg'],
                          accelerator="auto",   # uses GPU if available
                          batch_size= best_params['batch_size']
                          )
    model = model.add_country_holidays(country_name="US")
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
    model.fit(train, freq="D", progress="off")

    # build dataframe containing future regressors
    future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    forecast = model.predict(future)

    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_true = test["y"].values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"fold": i, "rmse": rmse, "mape": mape})

neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
neural_prophet_results_df

Training: 0it [00:00, ?it/s]

Predicting: 11it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 12it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

...

Training: 0it [00:00, ?it/s]

Predicting: 14it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Predicting: 15it [00:00, ?it/s]

,fold,rmse,mape
0,0,1.193401,0.604899
1,1,1.000905,0.760439
2,2,1.092501,0.617643
3,3,1.038960,0.465065
4,4,0.419753,0.242122
5,5,0.607598,0.433935
6,6,0.910022,0.509738
7,7,1.079373,0.508723
8,8,0.492126,0.284309
9,9,1.505063,0.597763


## Final Model

In [19]:
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']


lags_for_regressed_features = dict()
lags_for_regressed_features['apparent_temperature_max'] = best_params['lag_temp_max']
lags_for_regressed_features['apparent_temperature_min'] = best_params['lag_temp_min']
lags_for_regressed_features['snowfall_sum'] = best_params['lag_snowfall']


model = NeuralProphet(yearly_seasonality=True, 
                          weekly_seasonality=True, 
                          batch_size = best_params['batch_size'],
                          ar_reg=best_params['ar_reg'],
                          learning_rate = best_params['learning_rate'],
                          epochs = best_params['epochs'],
                          accelerator="auto",   # uses GPU if available
                          n_lags= best_params['n_lags']
                          )
model = model.add_country_holidays(country_name="US")
for column in regressed_features:
    model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])
        
model.fit(rs, freq="D", progress="off")

new_rs = rs.copy()

new_rs = new_rs.drop(columns=['y'])

forecast = model.predict(rs)

Training: 0it [00:00, ?it/s]

Predicting: 15it [00:00, ?it/s]

In [20]:
model.plot(forecast)

ERROR - (NP.plotly.plot) - plotly-resampler is not installed. Please install it to use the resampler.
